# 04 · Confidence band by root finding

`find_confidence_band` locates the band edges directly: it brackets the
outer reach with the widest level (2σ), then pins each level's edge by
geometric bisection. Far fewer Monte-Carlo evaluations than a dense grid.

Key knobs (see lesson 08 for tuning):
- `num_pseudo_data` — bracketing sample size (≥50 for 2σ)
- `n_pseudo_edge`   — edge bisection sample size (≥500 to converge)
- `step`            — bracket expansion factor (use 1.5)
- `rel_tol`         — edge resolution
- `seed`            — fix for reproducibility

In [1]:
import warnings; warnings.simplefilter('ignore')
from neutrino_analysis_band import NeutrinoAnalysis

a = NeutrinoAnalysis(background_scenario='flat', intervals='180',
                     GeV=0.32e16, solver='osqp', T=3)
a.optimize(a.data_vector)

band = a.find_confidence_band(
    fixed_index=20,
    levels=(0.678, 0.90, 0.954),
    num_pseudo_data=30, n_pseudo_edge=120,   # small for a quick demo
    step=1.5, rel_tol=0.05, seed=42, verbose=True,
)

[band idx=20] v0=9.1380e+11 (phys), bracketing with step=1.5
  upper bracket: 1.3707e+12
  lower bracket: 6.0920e+11
  level 0.678: [7.2768e+11, 1.0915e+12] (phys)
  level 0.900: [6.9172e+11, 1.2080e+12] (phys)
  level 0.954: [6.9172e+11, 1.2708e+12] (phys)


In [2]:
# The result is nested: tighter levels sit inside wider ones.
print('best fit (phys) = %.3e' % band['best_fit_physical'])
for lv in band['levels']:
    lo, hi = band['band_physical'][lv]
    print('  %.3f : [%.3e, %.3e]' % (lv, lo, hi))

best fit (phys) = 9.138e+11
  0.678 : [7.277e+11, 1.092e+12]
  0.900 : [6.917e+11, 1.208e+12]
  0.954 : [6.917e+11, 1.271e+12]


**Note:** with small `n_pseudo_edge` the edges carry Monte-Carlo noise.
Increase it (≥500) for publication numbers — lesson 08 shows the
convergence behaviour.